# QF 627 Programming and Computational Finance
### Lesson 09 | Unsupervised Learning, Clustering, and Pairs Trading

> Hi, Team 👋 

> Last week we learned how to use an unsupervised learning framework of dimensionality reduction (in particular, PCA) to solve an asset allocation and portfolio management problem. This week we will continue to see how useful clustering can be for common questions in quantitative finance.

> We will start with a quick lesson on how k-means clustering is operated. This will give you an initial understanding of clustering algorithms. We will then use clustering for paired trading. After our discussion leading session, we will continue to see how to use clustering to identify support and resistance levels.

data wrangling

rule based a-z

supervised

unsupervised 
- PCA, 
- kmeans clustering 
- algo clustering

### Activation of necessary modules

In [72]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

import pandas_datareader as dr
from pandas_datareader import data as pdr

import datetime
import yfinance as yf

import warnings

#### Setting plotting and display options

In [73]:
np.set_printoptions(precision = 3)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

%matplotlib inline

warnings.filterwarnings("ignore")

    PROBLEM STATEMENT
    
> Our goal in this case study is to perform clustering analysis on S&P500 stocks, and to come up with pairs, for a pairs trading strategy.

> The stocks data are obtained using pandas_datareader from Yahoo Finance. It includes price data from January 2008 to October 11th of 2019.

## Pairs Trading Using Clustering

> Pairs trading with clustering is an advanced market-neutral strategy that leverages data-driven techniques to identify tradable pairs. Let's dive deeper into its rationale and execution:

### Basic Pairs Trading

> Pairs trading is a market-neutral trading strategy designed to exploit anomalies between two or more securities.

> The foundational idea is:

- If two stocks historically move in tandem and suddenly diverge, they're expected to revert to their historical norm.
- Strategy: Short the outperforming stock and buy the underperforming one.

### Why Use Clustering?

Identifying which pairs of stocks move congruently is a challenge. This is where clustering offers its utility.

### What is Clustering?

Clustering is an unsupervised machine learning technique that groups similar items:

- For stocks, it means clustering them based on similar price movements.
- Stocks within the same cluster are anticipated to have similar movements.

### Benefits of Clustering in Pairs Trading

- **Identification of Pairs**: Automates the process of finding stocks that move similarly.
- **Dynamic Adaptation**: Allows traders to adapt to changing market dynamics by periodically re-running the clustering algorithm.
- **Diversification**: Identifies multiple pairs across clusters, diversifying the trading strategy.
- **Risk Management**: Diversified pairs trading can cushion against sector-specific downturns.

### Oft-used Clustering Techniques

- **K-Means Clustering**: Partitions data into \( K \) distinct, non-overlapping clusters.
- **Hierarchical Clustering**: Constructs a tree of clusters, suitable for nested relationships.
- **DBSCAN**: Ideal for clusters of varying shapes.

### Workflow of Pairs Trading

    Step 1. Choose a historical window of stock price data.
    Step 2. Compute returns (and volatility) for each stock.
    Step 3. Apply a clustering algorithm based on these returns (and volatilities).
    Step 4. Identify tradable pairs or groups within each cluster.
    Step 5. Monitor pairs for deviations and trade based on historical norms.

### Towards Data-driven Trading Strategy Building

> Pairs trading using clustering is a methodical, data-centric approach to pinpoint and capitalize on price discrepancies between securities. Clustering algorithms unveil latent relationships in the data and acclimatize to evolving market conditions.


### Load Packages for ML

In [74]:
import pandas as pd
from pandas import read_csv, set_option

In [75]:
from pandas.plotting import scatter_matrix
import seaborn as sns

from sklearn.preprocessing import StandardScaler

In [76]:
from sklearn.cluster import KMeans, AgglomerativeClustering,AffinityPropagation, DBSCAN
from scipy.cluster.hierarchy import fcluster
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import pdist
from sklearn.metrics import adjusted_mutual_info_score
from sklearn import cluster, covariance, manifold

In [77]:
import matplotlib.ticker as ticker
from itertools import cycle

### Step 1: IMPORT

In [79]:
dataset = read_csv("https://talktoroh.com/s/sp500.csv", 
                   index_col = 0)

In [80]:
type(dataset)

pandas.core.frame.DataFrame

### Step 2: Data Wrangling and Exploratory Data Analysis

#### Descriptive Statistics

In [81]:
dataset.shape

(448, 502)

In [10]:
# dataset.head(3)

In [11]:
# dataset.tail(3)

In [12]:
# dataset.describe()

#### Data Visualization?

> We will take a detailed look into visualization after clustering.

#### Data Preparation

> `Data Cleaning`

> We check for the NAs in the rows, and either drop them or fill them with the mean of the column.

In [13]:
# # Checking for any null values and removing missing values

# print("Missing Values? =", 
#       dataset
#           .isnull()
#           .values
#           .any()
#      )

> Delete any columns where more than 30% of the values are missing.

In [14]:
# missing_fractions = \
#     dataset \
#     .isnull() \
#     .mean() \
#     .sort_values(ascending = False)

In [15]:
# missing_fractions.head(10)

In [16]:
# drop_list =\
#     sorted(list(missing_fractions
#                 [missing_fractions > 0.3]
#                 .index)
#            )

In [17]:
# dataset =\
#     dataset \
#     .drop(labels= drop_list, 
#           axis=1)

In [18]:
# dataset.shape[1] == 502 - 4

> As there are null values, drop the rows containing the null values.

In [19]:
# # Fill the missing values with the last value available in the dataset. 

# dataset = dataset.fillna(method = "ffill")
# dataset.head()

#### Data Transformation

> For the purpose of clustering, we will use annual returns and variance as the variables as they are indicators of a stock’s performance and its volatility. 

> Let us prepare the return and volatility variables from the data.

In [20]:
# #Calculate average annual percentage return and volatilities over a theoretical one year period

# returns =\
# (
#     dataset
#     .pct_change()
#     .mean() 
#     * 252
# )

# returns = pd.DataFrame(returns)

# returns.columns = ["Returns"]

In [21]:
# returns["Volatility"] =\
# (    
#      dataset
#     .pct_change()
#     .std() 
#     * np.sqrt(252)
# )

# data = returns.copy()

In [22]:
# returns

In [23]:
# # You may format the data as a numpy array to feed into the K-Means algorithm

# data =\
# ( 
#     np 
#     .asarray([np.asarray(returns['Returns']),
#               np.asarray(returns['Volatility'])
#              ]
#             )
#     .T
# )

In [24]:
# data

> All the variables should be on the same scale before applying clustering. 

> Otherwise, a feature with large values will dominate the result. 

> We use `StandardScaler` in sklearn to standardize the dataset’s features onto a unit scale (mean = 0 and variance = 1).

In [25]:
# from sklearn.preprocessing import StandardScaler

In [26]:
# scaler = StandardScaler().fit(data)

In [27]:
# rescaledDataset =\
# (
#     pd
#     .DataFrame(scaler.fit_transform(data),
#                columns = data.columns, 
#                index = data.index)
# )
# rescaledDataset.head()

In [28]:
# # summarize transformed data
# X = rescaledDataset
# X.head()

> The parameters to the clusters are the indices, and the variables used in the clustering are the columns. So the data is in the right format to be fed to the clustering algorithms.

# Step 3: Model (with `Clustering`)

> We will look at the following models:

#### 1. KMeans
#### 2. Hierarchical Clustering (Agglomerative Clustering)
#### 3. Affinity Propagation 

### K-Means Clustering


#### Finding optimal number of clusters

In this step we look at the following metrices:

1. Sum of square errors (SSE) within clusters
2. Silhouette score.

In [29]:
# distorsions = []

# max_loop = 20

# for k in range(2, max_loop):
#     kmeans = KMeans(n_clusters = k)
#     kmeans.fit(X)
#     distorsions.append(kmeans.inertia_)
    
# fig = plt.figure(figsize=(16, 8))

# plt.plot(range(2, max_loop), 
#          distorsions)

# plt.xticks([i for i in range(2, max_loop)], 
#            rotation=75)

# plt.grid(True)

> Inspecting the sum of squared errors chart, it appears that the elbow `kink` occurs in five or six clusters for this data. 

> Certainly, we can see that as the number of clusters increases beyond six, the sum of the square of errors within clusters reaches a plateau.

#### Silhouette score

In [30]:
# from sklearn import metrics

In [31]:
# silhouette_score = []

# for k in range(2, max_loop):
#         kmeans = KMeans(n_clusters = k,  
#                         random_state = 627, 
#                         n_init = 10)
#         kmeans.fit(X)        
#         silhouette_score.append(metrics.silhouette_score(X, kmeans.labels_, random_state = 627)
#                                )
        
# fig = plt.figure(figsize=(16, 10)
#                 )

# plt.plot(range(2, max_loop), silhouette_score)

# plt.xticks([i for i in range(2, max_loop)], 
#            rotation=75)

# plt.grid(True)

> On the silhouette score chart, there are various parts of the graph where a kink can be seen. Since there is not much difference in SSE after six clusters, we would prefer to have six clusters in the k-means model.

#### Clustering and Visualisation

> Let us build the k-means model with six clusters and visualize the results.

In [32]:
# nclust = 6

In [33]:
# # Fit with k-means
# k_means = cluster.KMeans(n_clusters=nclust)
# k_means.fit(X)

In [34]:
# !pip install threadpoolctl==3.1.0

In [35]:
# !pip install numpy==1.21.4

In [36]:
# import numpy as np

In [37]:
# import threadpoolctl 

In [38]:
# !pip install -U scikit-learn

In [39]:
# import sklearn

# print(sklearn.show_versions()
#      )

In [40]:
# # Extracting labels 

# target_labels = k_means.predict(X)

In [41]:
# target_labels

> Visualizing how your clusters are formed is not easy, when the number of variables or dimensions in your dataset is very large. 

> One of the methods of visualizing a cluster is two-dimensional space.

In [42]:
# centroids = k_means.cluster_centers_

# fig = plt.figure(figsize=(16,10)
#                 )

# ax = fig.add_subplot(111)

# scatter =\
# (    
#     ax
#     .scatter(X.iloc[ : ,0], 
#              X.iloc[ : ,1], 
#              c = k_means.labels_, 
#              cmap = "rainbow", 
#              label = X.index)
# )

# ax.set_title("k-Means results")
# ax.set_xlabel("Average Return")
# ax.set_ylabel("Volatility")

# plt.colorbar(scatter)

# plt.plot(centroids[:,0], 
#          centroids[:,1], 
#          "sg", 
#          markersize = 15, 
#          color = "black")

Let us check the elements of the clusters

In [43]:
# # show number of stocks in each cluster

# clustered_series =\
# (
#     pd
#     .Series(index = X.index, 
#             data = k_means
#                    .labels_
#                    .flatten()
#             )
# )

In [44]:
# # clustered stock with its cluster label
# clustered_series_all =\
# (    
#     pd
#     .Series(index=X.index, 
#             data=k_means.labels_.flatten()
#             )
# )

# clustered_series = clustered_series[clustered_series != -1]

In [45]:
# plt.figure(figsize=(16,8)
#           )

# plt.barh(
#     range(len(clustered_series.value_counts()
#              )
#          ), # cluster labels, y axis
#     clustered_series.value_counts()
# )

# plt.title("Cluster Member Counts")
# plt.xlabel("Stocks in Cluster")
# plt.ylabel("Cluster Number")

# plt.show()

In [46]:
# plt.figure(figsize=(16,8)
#           )

# counts =\
# (    
#     clustered_series
#     .value_counts()
#     .sort_index()
# )

# plt.barh(counts.index, 
#          counts)

# plt.title("Cluster Member Counts")
# plt.xlabel("Stocks in Cluster")
# plt.ylabel("Cluster Number")

# plt.show()

> The number of stocks in a cluster range from around 40 to 120. Although the distribution is not equal, we have an adequate number of stocks in each cluster.

### Hierarchical Clustering (Agglomerative Clustering)

> In the first step we look at the hierarchy graph and check for the number of clusters.

#### Building Hierarchy Graph/ Dendogram

> The hierarchy class has a dendrogram method which takes the value returned by the linkage method of the same class. The linkage method takes the dataset and the method to minimize distances as parameters. We use “ward” as the method since it minimizes the variants of distances between the clusters.

In [47]:
# from scipy.cluster.hierarchy import dendrogram, linkage, ward

In [48]:
# #Calulate linkage
# Z = linkage(X, 
#             method = "ward")
# Z[0]

> The best way to visualize an agglomerate clustering algorithm is through a dendogram, which displays a cluster tree. 

> The tree’s leaves are the individual stocks and the root is the final single cluster. 

> The `distance` between each cluster is shown on the y-axis; the longer the branches are, the less correlated two clusters are.

In [49]:
# # Plot Dendogram

# plt.figure(figsize=(18, 10)
#           )
# plt.title("Stocks Dendograms")

# dendrogram(Z, labels = X.index)

# plt.show()

> Once one big cluster is formed, the longest vertical distance without any horizontal line passing through it is selected and a horizontal line is drawn through it. 

> The number of vertical lines this newly created horizontal line passes is equal to the number of clusters.

In [50]:
# distance_threshold = 13

# clusters = fcluster(Z, distance_threshold, criterion='distance')

# chosen_clusters = pd.DataFrame(data=clusters, 
#                                columns=['cluster']
#                               )

# chosen_clusters['cluster'].unique()

> We then select the distance threshold to cut the dendrogram to obtain the selected clustering level. 

> The output is the cluster labeled for each row of data. 

> As expected from the dendrogram, a cut at 13 gives us four clusters.

#### Clustering and Visualisation

In [51]:
# nclust = 4

# hc = AgglomerativeClustering(n_clusters = nclust, 
#                              affinity = "euclidean", 
#                              linkage = "ward")

# clust_labels1 = hc.fit_predict(X)

In [52]:
# fig = plt.figure(figsize=(16,10)
#                 )

# ax = fig.add_subplot(111)

# scatter = ax.scatter(X.iloc[:,0], 
#                      X.iloc[:,1], 
#                      c = clust_labels1, 
#                      cmap = "rainbow")

# ax.set_title("Hierarchial")
# ax.set_xlabel("Average Return")
# ax.set_ylabel("Volatility")

# plt.colorbar(scatter)

> Similar to the plot of k-means clustering, we see that there are some distinct clusters
separated by different colors. 

### Affinity Propagation

In [53]:
# ap = AffinityPropagation()

# ap.fit(X)

# clust_labels2 = ap.predict(X)

In [54]:
# fig = plt.figure(figsize=(16,10)
#                 )
# ax = fig.add_subplot(111)

# scatter = ax.scatter(X.iloc[:,0], 
#                      X.iloc[:,1], 
#                      c = clust_labels2, 
#                      cmap = "rainbow")

# ax.set_title("Affinity")
# ax.set_xlabel("Average Return")
# ax.set_ylabel("Volatility")
             
# plt.colorbar(scatter)

Similar to the plot of k-means clustering, we see that there are some distinct clusters separated by different colors. 

<a id='5.3.1'></a>
### 5.3.1 Cluster Visualisation

In [55]:
# cluster_centers_indices = ap.cluster_centers_indices_
# labels = ap.labels_

In [56]:
# no_clusters = len(cluster_centers_indices)
# print("Estimated number of clusters: %d" % no_clusters)

# # Plot exemplars

# X_temp = np.asarray(X)

# plt.close("all")
# plt.figure(1)
# plt.clf()

# fig = plt.figure(figsize=(18,10)
#                 )
# colors = cycle("bgrcmykbgrcmykbgrcmykbgrcmyk") # The sequence "bgrcmyk" (blue, green, red, cyan, magenta, yellow, black) 
#                                                # is repeated four times

# for k, col in zip(range(no_clusters), colors):
    
#     class_members = labels == k
#     cluster_center = X_temp[cluster_centers_indices[k]]
    
#     plt.plot(X_temp[class_members, 0], X_temp[class_members, 1], col + ".")
    
#     plt.plot(cluster_center[0], cluster_center[1], 
#              "o", 
#              markerfacecolor = col, 
#              markeredgecolor = "k", 
#              markersize = 14)
    
#     for x in X_temp[class_members]:
#         plt.plot([cluster_center[0], x[0]], [cluster_center[1], x[1]], col)

# plt.show()

In [57]:
# # show number of stocks in each cluster
# clustered_series_ap = pd.Series(index=X.index, data=ap.labels_.flatten()
#                                )

# # clustered stock with its cluster label
# clustered_series_all_ap = pd.Series(index=X.index, data=ap.labels_.flatten())
# clustered_series_ap = clustered_series_ap[clustered_series != -1]

In [58]:
# plt.figure(figsize=(16,10)
#           )

# plt.barh(
#     range(len(clustered_series_ap.value_counts()
#              )
#          ), # cluster labels, y axis
#     clustered_series_ap.value_counts()
# )

# plt.title("Cluster Member Counts")
# plt.xlabel("Stocks in Cluster")
# plt.ylabel("Cluster Number")

# plt.show()

### Cluster Evaluation

> If the ground truth labels are not known, an evaluation must be performed using the model itself. 

> The Silhouette Coefficient (`sklearn.metrics.silhouette_score`) is an example of such an evaluation. 

> A higher Silhouette Coefficient score means a model with better defined clusters. 

> The Silhouette Coefficient is defined for each sample and is composed of two scores:

In [59]:
# from sklearn import metrics

In [60]:
# print("km", metrics.silhouette_score(X, k_means.labels_, 
#                                      metric='euclidean')
#      )

# print("hc", metrics.silhouette_score(X, hc.fit_predict(X), 
#                                      metric='euclidean')
#      )

# print("ap", metrics.silhouette_score(X, ap.labels_, 
#                                      metric='euclidean')
#      )

> Let's go with `affinity propagation` here.

> We use 27 clusters, as specified by this clustering method.

### Visualising the return within a cluster

> To understand the intuition behind clustering, let us visualize the results of the clusters.

In [61]:
# # All stock with its cluster label (including -1)
# clustered_series = pd.Series(index = X.index, data = ap.fit_predict(X).flatten()
#                             )

# # Clustered stock with its cluster label

# clustered_series_all = pd.Series(index = X.index, data = ap.fit_predict(X).flatten()
#                                 )

# clustered_series = clustered_series[clustered_series != -1]

In [62]:
# # Get the number of stocks in each cluster
# counts = clustered_series_ap.value_counts()

# # Let's visualize some clusters
# cluster_vis_list = list(counts[(counts < 25) & (counts > 1)].index)[::-1]
# cluster_vis_list

In [63]:
# CLUSTER_SIZE_LIMIT = 9999

# counts = clustered_series.value_counts()

# ticker_count_reduced = counts[(counts>1) & (counts <= CLUSTER_SIZE_LIMIT)]

# print ("Clusters formed: %d" % len(ticker_count_reduced)
#       )
# print ("Pairs to evaluate: %d" % (ticker_count_reduced*(ticker_count_reduced-1)
#                                  ).sum()
#       )

In [64]:
# # plot a handful of the smallest clusters
# plt.figure(figsize=(16,10)
#           )
# cluster_vis_list[0:min(len(cluster_vis_list), 4)]

In [65]:
# for clust in cluster_vis_list[0:min(len(cluster_vis_list), 4)]:
    
#     tickers = list(clustered_series[clustered_series == clust].index)
    
#     means = np.log(dataset.loc[:"2018-02-01", tickers].mean())
    
#     data = np.log(dataset.loc[:"2018-02-01", tickers]).sub(means)
    
#     data.plot(title='Stock Time Series for Cluster %d' % clust)
    
# plt.show()

> Looking at the charts above, in all the clusters with a small number of stocks, we see similar movement of the stocks. 
> This corroborates the effectiveness of the clustering technique.

## `Pairs Selection`

> Cointegration and Pair Selection Function

### `Cointegration` vs. Correlation

>  ***Correlation***

Correlation measures the linear relationship between two variables. The Pearson correlation coefficient between two variables \( X \) and \( Y \) is defined as:

$$
\rho_{X,Y} = \frac{\text{cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Where:
- \( \text{cov}(X, Y) \) is the covariance between \( X \) and \( Y \).
- \( \sigma_X \) and \( \sigma_Y \) are the standard deviations of \( X \) and \( Y \) respectively.

> ***Cointegration***

Cointegration deals with the relationship between non-stationary time series. Two time series \( X_t \) and \( Y_t \) are said to be cointegrated if:

1. Both \( X_t \) and \( Y_t \) are non-stationary (typically, they have a unit root).
2. A linear combination exists such that \( Z_t = aX_t + bY_t \) is stationary.

If \( X_t \) and \( Y_t \) are cointegrated, there exists coefficients \( a \) and \( b \) (where \( b \neq 0 \)) such that the time series \( Z_t \) is stationary.

### Why Use Cointegration in Pairs Trading?

In pairs trading, the goal is to exploit deviations from a long-term equilibrium between two securities. While correlation can indicate a linear relationship, it doesn't ensure that the relationship is stable over time.

Cointegration, on the other hand, implies a long-term equilibrium relationship. If two securities are cointegrated, it means that even if they drift apart in the short term, they will revert back to a long-term equilibrium. This reversion is the essence of pairs trading.

All in all:
- **Correlation**: Measures the strength and direction of linear relationships.
- **Cointegration**: Focuses on the long-term equilibrium relationship between time series, which is crucial for pairs trading strategies.

Using cointegration in pairs trading provides more assurance of the existence of a stable, long-term relationship between the paired securities, which is the foundation of the strategy.


In [66]:
# def find_cointegrated_pairs(data, significance=0.05):
#     # Get the number of columns in the data (i.e., number of securities)
#     n = data.shape[1]
    
#     # Initialize a matrix filled with zeros to store cointegration scores
#     score_matrix = np.zeros((n, n))
    
#     # Initialize a matrix filled with ones to store p-values of the cointegration tests
#     pvalue_matrix = np.ones((n, n))
    
#     # Extract the column names (security names) from the data
#     keys = data.keys()
    
#     # List to store pairs of securities that are cointegrated
#     pairs = []
    
#     # Double loop to go through each combination of securities
#     for i in range(n):  
#         for j in range(i+1, n):
            
#             # Extract the time series data for the two securities in consideration
#             S1 = data[keys[i]]
#             S2 = data[keys[j]]
            
#             # Perform the cointegration test between the two securities
#             result = coint(S1, S2)
            
#             # Extract the score (test statistic) and p-value from the result
#             score = result[0]
#             pvalue = result[1]
            
#             # Store the score and p-value in their respective matrices
#             score_matrix[i, j] = score
#             pvalue_matrix[i, j] = pvalue
            
#             # If the p-value is less than the significance level, 
#             # then the pair is considered cointegrated and added to the pairs list
#             if pvalue < significance:
#                 pairs.append((keys[i], keys[j]))

#     # Return the score matrix, p-value matrix, and the list of cointegrated pairs
#     return score_matrix, pvalue_matrix, pairs

In [67]:
# from statsmodels.tsa.stattools import coint

In [68]:
# cluster_dict = {}

# for i, which_clust in enumerate(ticker_count_reduced.index):
    
#     tickers = clustered_series[clustered_series == which_clust].index   
    
#     score_matrix, pvalue_matrix, pairs = find_cointegrated_pairs(dataset[tickers]
#                                    )
#     cluster_dict[which_clust] = {}
#     cluster_dict[which_clust]["score_matrix"] = score_matrix
#     cluster_dict[which_clust]["pvalue_matrix"] = pvalue_matrix
#     cluster_dict[which_clust]["pairs"] = pairs

In [69]:
# pairs = []
# for clust in cluster_dict.keys():
#     pairs.extend(cluster_dict[clust]["pairs"])

In [70]:
# print ("Number of pairs found : %d" % len(pairs)
#       )
# print ("In those pairs, there are %d unique tickers." % len(np.unique(pairs)
#                                                            )
#       )

In [71]:
# pairs

### What We Learned

> Clustering techniques do not directly help in stock trend prediction. However, they can be used in portfolio construction to find the right pairs. This eventually helps in risk mitigation, and one can achieve superior risk-adjusted returns.

> We have looked at methods of finding the appropriate number of clusters in k-means, and have built a hierarchy graph in hierarchical clustering. A next step in this case study would be to explore and backtest various long/short trading strategies with pairs of stocks from the groupings of stocks.

> Clustering can be used to divide stocks into groups with`similar characteristics` for many other kinds of trading strategies. It can also help in portfolio construction, to ensure we select stocks with sufficient diversification between them.

> Hierarchical clustering is a valuable tool for identifying potential pairs in pairs trading strategies, especially with small-scale data. By grouping assets based on their return similarities, traders can systematically find pairs that exhibit correlated behavior.

> However, it's crucial to complement clustering with additional statistical tests like cointegration analysis to confirm the robustness of the identified pairs. With careful implementation and risk management, hierarchical clustering can enhance the effectiveness of pairs trading strategies.

> `Thank you for working with the script, Team 👍`